In [21]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm # Or standard tqdm if not in notebook
import time
import copy # To save best model state
import random

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchaudio.transforms as T # For SpecAugment

In [16]:
# Audio Processing Parameters
SAMPLE_RATE = 32000
DURATION = 5  # seconds
N_MELS = 128
FMIN = 20
FMAX = 14000 # Birds typically don't vocalize much higher than this
HOP_LENGTH = int(SAMPLE_RATE * 0.01)  # 10ms hop: 32000 * 0.01 = 320
N_FFT = int(SAMPLE_RATE * 0.025)     # 25ms window: 32000 * 0.025 = 800, round to 1024 for FFT efficiency
N_FFT = 1024 # Or 2048 as you had, both are fine. Let's try 1024.
NUM_CLASSES = 206
# Augmentation Parameters
APPLY_AUGMENTATION_PROB = 0.6 # Probability of applying augmentations to a training sample
NOISE_LEVEL = 0.005
TIME_SHIFT_FRACTION = 0.2 # Max fraction of duration to shift
PITCH_SHIFT_STEPS = 2 # Max semitones for pitch shift
# Model & Training Parameters
MODEL_NAME = "efficientnet_b0" # e.g., "efficientnet_b0", "efficientnet_b2"
BATCH_SIZE = 32
EPOCHS = 25 # Pre-trained models can benefit from more epochs with good augmentation
LEARNING_RATE = 3e-4 # Often a good starting LR for AdamW with pre-trained models
WEIGHT_DECAY = 0.01 # For AdamW
VALIDATION_SPLIT = 0.2
RANDOM_SEED = 42
NUM_WORKERS = 2 # os.cpu_count() // 2 if you have many cores
LABEL_SMOOTHING = 0.1 # Set to 0 to disable
PRELOAD_DATA_IN_RAM = True # Set to True if you have enough RAM and want faster epochs after initial load
# --- Set Seed for Reproducibility ---
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.

seed_everything(RANDOM_SEED)
TARGET_LENGTH_SAMPLES = SAMPLE_RATE * DURATION
N_FRAMES = int(np.ceil(TARGET_LENGTH_SAMPLES / HOP_LENGTH))
print(f"Expected Spectrogram Shape (H, W): ({N_MELS}, {N_FRAMES})")
# --- Determine Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Expected Spectrogram Shape (H, W): (128, 500)
Using device: cpu


In [17]:
# --- 6. Build Custom Model with EfficientNet Backbone ---
print(f"\nBuilding custom model with {MODEL_NAME} backbone...")

# Define a custom model class that uses EfficientNet as a backbone
class BirdClassifier(nn.Module):
    def __init__(self, backbone_name, num_classes, pretrained=True):
        super(BirdClassifier, self).__init__()
        
        # Get the EfficientNet backbone
        if backbone_name == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=weights)
            backbone_features = backbone.features
            num_ftrs = 1280  # For EfficientNet B0
        
        # Extract the feature extractor (everything except the classifier)
        self.backbone = backbone_features
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Custom classifier with an additional hidden layer
        self.classifier = nn.Sequential(
            nn.Linear(1280, 1024),  # First hidden layer
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),  # Second hidden layer
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 206)  # Final layer with 206 classes
        )
        
        # Softmax activation for final layer (not included in training since CrossEntropyLoss has it)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # Extract features using the backbone
        x = self.backbone(x)
        
        # Global Average Pooling
        x = self.gap(x)
        x = torch.flatten(x, 1)
        
        # Classification head
        x = self.classifier(x)
        
        # Note: We don't apply softmax during training since CrossEntropyLoss includes it
        # We only apply it during inference or when raw probabilities are needed
        return x
    
    def predict_proba(self, x):
        # For getting probabilities during inference
        logits = self.forward(x)
        return self.softmax(logits)

# Initialize the model
model = BirdClassifier(MODEL_NAME, NUM_CLASSES, pretrained=False)
model = model.to(device)

# Fine-tuning strategy
# Option 1: Unfreeze all layers for end-to-end fine-tuning
for param in model.parameters():
    param.requires_grad = True
print("Fine-tuning: Training all layers.")

# Option 2: Freeze backbone and train only classifier for the first few epochs
# Comment this section if you want to train all layers from the beginning
# for name, param in model.backbone.named_parameters():
#     param.requires_grad = False
# print("Fine-tuning: Only training the classifier initially.")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")



Building custom model with efficientnet_b0 backbone...
Fine-tuning: Training all layers.
Total model parameters: 5,952,842
Trainable parameters: 5,952,842


In [18]:
# --- 9. Evaluation (using the best saved model) ---
print("\nEvaluating the best custom model on the validation set...")

# Load the best model state
best_model_path = '/kaggle/input/new-efficientnet-birdclef-25/best_custom_efficientnet_b0_model.pth'
if os.path.exists(best_model_path):
    print(f"Loading best model from '{best_model_path}'")
    # Create an instance of your custom BirdClassifier class
    eval_model = BirdClassifier(MODEL_NAME, NUM_CLASSES, pretrained=False)
    eval_model.load_state_dict(torch.load(best_model_path, map_location=device))
    eval_model = eval_model.to(device)  # Move to device
    eval_model.eval()  # Set to evaluation mode
else:
    print(f"Warning: No best model found at '{best_model_path}'. Evaluating the final model state (if available).")
    # Use the model currently in memory (last epoch state)
    eval_model = model
    eval_model.eval()


Evaluating the best custom model on the validation set...
Loading best model from '/kaggle/input/new-efficientnet-birdclef-25/best_custom_efficientnet_b0_model.pth'


/tmp/ipykernel_31/851123796.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  eval_model.load_state_dict(torch.load(best_model_path, map_location=device))


In [26]:
# --- 10 create submission
import os
import librosa
import numpy as np
import pandas as pd
import torch.nn.functional as F
# Set seed
np.random.seed(42)

# Class labels from train audio
class_labels = sorted(os.listdir('/kaggle/input/birdclef-2025/train_audio/'))

# List of test soundscapes (only visible during submission)
test_soundscape_path = '/kaggle/input/birdclef-2025/test_soundscapes/'
test_soundscapes = [os.path.join(test_soundscape_path, afile) for afile in sorted(os.listdir(test_soundscape_path)) if afile.endswith('.ogg')]
# Open each soundscape and make predictions for 5-second segments
# Use pandas df with 'row_id' plus class labels as columns
predictions = pd.DataFrame(columns=['row_id'] + class_labels)


In [27]:

for soundscape in test_soundscapes:

    # Load audio
    sig, rate = librosa.load(path=soundscape, sr=None)

    # Split into 5-second chunks
    chunks = []
    for i in range(0, len(sig), rate*5):
        chunk = sig[i:i+rate*5]
        chunks.append(chunk)
        
            
            
    # Make predictions for each chunk
    for i, chunk in enumerate(chunks):
            
        # Get row id  (soundscape id + end time of 5s chunk)      
        row_id = os.path.basename(soundscape).split('.')[0] + f'_{i * 5 + 5}'
            
        # Make prediction (let's use random scores for now)
        target_length = len(chunk)
    
        mel_spec = librosa.feature.melspectrogram(
            y=chunk, sr=32000, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, fmin=FMIN, fmax=FMAX
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
        # --- Normalization (Optional but recommended for pre-trained models) ---
        # Normalize to roughly [0, 1] or [-1, 1] range if needed,
        # or use ImageNet stats if model expects that.
        # Simple min-max scaling to [0, 1]:
        min_val = np.min(mel_spec_db)
        max_val = np.max(mel_spec_db)
        if max_val > min_val:
            mel_spec_db = (mel_spec_db - min_val) / (max_val - min_val)

        # Ensure consistent shape (especially time dimension) due to potential minor librosa variations
        if mel_spec_db.shape[1] < N_FRAMES:
            pad_width = N_FRAMES - mel_spec_db.shape[1]
            mel_spec_db = np.pad(mel_spec_db, ((0,0), (0, pad_width)), mode='constant', constant_values=0.)
        elif mel_spec_db.shape[1] > N_FRAMES:
            mel_spec_db = mel_spec_db[:, :N_FRAMES]

        mel_spec_resized = cv2.resize(mel_spec_db, (224, 224), interpolation=cv2.INTER_LINEAR)
            # Convert to PyTorch Tensor and add channel dimension
        spectrogram_tensor = torch.tensor(mel_spec_resized, dtype=torch.float32).unsqueeze(0)
                
            # Repeat channel 3 times to mimic RGB
        spectrogram_tensor_3channel = spectrogram_tensor.repeat(3, 1, 1)  # Output shape: (3, H, W)
        spectrogram_tensor_3channel = spectrogram_tensor_3channel.unsqueeze(0)
        scores = F.softmax(eval_model(spectrogram_tensor_3channel), dim=1)
            
        # Append to predictions as new row
        # Flatten the scores tensor and convert to a list of floats
        scores_list = scores.squeeze().detach().cpu().numpy().tolist()
        print(scores_list.index(max(scores_list)))
        # Create the DataFrame row
        new_row = pd.DataFrame([[row_id] + scores_list], columns=['row_id'] + class_labels)
        predictions = pd.concat([predictions, new_row], axis=0, ignore_index=True)
        

# Save prediction as csv
predictions.to_csv('submission.csv', index=False)
predictions.head()

,row_id,1139490,1192948,1194042,126247,1346504,134933,135045,1462711,1462737,...,yebfly1,yebsee1,yecspi2,yectyr1,yehbla2,yehcar1,yelori1,yeofly1,yercac1,ywcpar
